# Aula 14 - PySpark: ROW NUMBER

In [0]:
# Aula 14 - ROW NUMBER (numeração de linhas por partição)
# Objetivo: ranquear carros dentro de cada modelo, do mais barato ao mais caro
# Observação: o título original do notebook diz 'Aula 13' por engano — conteúdo é da Aula 14

# Imports de funções e tipos (Window é importado na última célula)
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Leitura do CSV + distinct() contra duplicatas
df_carros = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('encoding', 'utf-8')
    .load('/Workspace/Users/jcesarsantana215@gmail.com/Aulas PySpark/datasets/modelo_carro.csv', sep=',')
    .distinct()
)

# Normaliza o preço: tira '$' e converte para DOUBLE (exigido para ORDER BY numérico)
df_carros = df_carros.withColumn(
    'preco',
    regexp_replace(
        'preco', r'\$', '').cast(DoubleType())
)

display(df_carros)

In [0]:
# ROW_NUMBER via SQL: enumera linhas reiniciando a cada modelo (PARTITION BY)
# ORDER BY preco ASC dá row_number=1 ao mais barato de cada modelo
# Útil para 'top-N por grupo' (WHERE row_number = 1 pega o mais barato de cada modelo)
df_carros.createOrReplaceTempView('carros')

df_carros_sql = spark.sql(
"""
    SELECT
        *,
        ROW_NUMBER() OVER(PARTITION BY modelo_carro ORDER BY preco ASC) AS row_number
    FROM carros
""")
display(df_carros_sql)

In [0]:
%sql
-- Mesmo ROW_NUMBER com o magic %sql (visualização rápida)
-- PARTITION BY define o grupo; sem ele, a numeração seria global e contínua
SELECT
    *,
    ROW_NUMBER() OVER(PARTITION BY modelo_carro ORDER BY preco ASC) AS row_number
FROM carros

In [0]:
# ROW_NUMBER com a API PySpark usando Window
# Window.partitionBy(...).orderBy(...) espelha o OVER(PARTITION BY ... ORDER BY ...)
# withColumn adiciona a numeração como nova coluna 'row_number'
from pyspark.sql.window import Window

df_carros_spark = df_carros.withColumn(
    "row_number",
    row_number().over(
        Window.partitionBy('modelo_carro').orderBy('preco')
    )
)

display(df_carros_spark)